# 01 — Constraint Analysis (flying wing)

Matching chart: required shaft power per kilogram vs wing loading. Each
performance requirement becomes a curve; stall and hand-launch become
vertical limits. The feasible design space is above every curve and left
of every limit.

**Inputs:** `mission/mission_params.yaml`, `design_point.yaml`,
`airframe_params.yaml` (aspect ratio sets the induced-drag factor).

**Outputs:** maximum allowable W/S, required shaft power at the design point.


In [ ]:
"""Locate the repo root so `import uavdesign` works without `pip install -e .`."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "uavdesign").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

PLATFORM_DIR = REPO_ROOT / "platforms" / "flying_wing"
FIG_DIR = REPO_ROOT / "outputs" / "figures" / "flying_wing"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
with open(REPO_ROOT / "mission" / "mission_params.yaml") as f:
    mission = yaml.safe_load(f)
with open(PLATFORM_DIR / "design_point.yaml") as f:
    design = yaml.safe_load(f)
with open(PLATFORM_DIR / "airframe_params.yaml") as f:
    airframe = yaml.safe_load(f)

from uavdesign.atmosphere import G, isa_density, wing_loading_g_dm2, wing_loading_n_m2
from uavdesign.aero import Wing, DragPolar, FixedWingAero

alt_m = mission["mission"]["altitude_asl_m"]
rho = isa_density(alt_m, mission["environment"]["temperature_offset_c"])
wing = Wing.from_dict(airframe["wing"])
aero = design["aero"]
perf = design["performance"]
mass_kg = design["design_point"]["target_auw_kg"]
static_margin = design["design_point"]["static_margin"]
polar = DragPolar.from_oswald(aero["cd0"], wing.aspect_ratio, aero["oswald_e"], aero["cl_max"])

print(f"Site: {alt_m} m ASL, rho = {rho:.4f} kg/m^3")
print(f"Wing: S = {wing.area_dm2:.2f} dm^2, b = {wing.span_mm:.0f} mm, AR = {wing.aspect_ratio:.2f}")
print(f"Polar: CD0 = {polar.cd0}, k = {polar.k:.4f}, CLmax = {polar.cl_max}")


## Constraint curves


In [ ]:
from uavdesign.aero import constraint as con

eta_p = design["propulsion"]["prop_efficiency"]
ws = np.linspace(20, 160, 400)  # N/m^2

v_cr, v_cl, v_tn = perf["cruise_speed_m_s"], perf["climb_speed_m_s"], perf["turn_speed_m_s"]
curves = {
    f"cruise @ {v_cr:.0f} m/s": con.power_per_mass_w_kg(
        con.cruise_tw(ws, rho, v_cr, polar.cd0, polar.k), v_cr, eta_p),
    f"climb {perf['climb_rate_m_s']:.0f} m/s @ {v_cl:.0f} m/s": con.power_per_mass_w_kg(
        con.climb_tw(ws, rho, v_cl, polar.cd0, polar.k, perf["climb_rate_m_s"]), v_cl, eta_p),
    f"{perf['turn_bank_deg']:.0f} deg sustained turn @ {v_tn:.0f} m/s": con.power_per_mass_w_kg(
        con.turn_tw(ws, rho, v_tn, polar.cd0, polar.k, perf["turn_bank_deg"]), v_tn, eta_p),
}
envelope = np.max(np.vstack(list(curves.values())), axis=0)

def ws_limits(cl_max):
    return (
        con.stall_ws_max(rho, perf["max_stall_speed_m_s"], cl_max),
        con.launch_ws_max(rho, perf["launch_speed_m_s"], cl_max, perf["launch_margin"]),
    )

ws_stall, ws_launch = ws_limits(polar.cl_max)
ws_limit = min(ws_stall, ws_launch)
ws_design = mass_kg * G / wing.area_m2
p_design = float(np.interp(ws_design, ws, envelope))

print(f"Stall limit  (Vs <= {perf['max_stall_speed_m_s']} m/s):   W/S <= {ws_stall:6.1f} N/m^2 "
      f"({wing_loading_g_dm2(ws_stall):.1f} g/dm^2)")
print(f"Launch limit ({perf['launch_speed_m_s']} m/s, {perf['launch_margin']} Vs): W/S <= {ws_launch:6.1f} N/m^2 "
      f"({wing_loading_g_dm2(ws_launch):.1f} g/dm^2)")
print(f"Design point: W/S = {ws_design:.1f} N/m^2 ({wing_loading_g_dm2(ws_design):.1f} g/dm^2), "
      f"needs {p_design:.0f} W/kg shaft")
print("Design point is", "FEASIBLE" if ws_design <= ws_limit else "OUTSIDE the stall/launch limit")


## Matching chart

Dotted verticals show how the binding W/S limit moves with CLmax — the
single most uncertain input for a reflexed flying-wing section.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for label, y in curves.items():
    ax.plot(ws, y, label=label)

top = float(np.nanmax(envelope[ws <= 160])) * 1.15
feasible = ws <= ws_limit
ax.fill_between(ws[feasible], envelope[feasible], top, color="C2", alpha=0.12, label="feasible")
ax.axvline(ws_stall, color="k", lw=1.5, label=f"stall limit (CLmax {polar.cl_max})")
ax.axvline(ws_launch, color="k", lw=1.5, ls="--", label="hand-launch limit")
for cl in aero["cl_max_sweep"]:
    if cl == polar.cl_max:
        continue
    lim = min(ws_limits(cl))
    ax.axvline(lim, color="grey", lw=0.8, ls=":")
    ax.text(lim, top * 0.97, f"{cl}", rotation=90, va="top", ha="right", fontsize=8, color="grey")
ax.plot(ws_design, p_design, "o", color="C3", ms=9, label="baseline design")

ax.set_xlim(ws[0], ws[-1])
ax.set_ylim(0, top)
ax.set_xlabel("Wing loading W/S [N/m^2]")
ax.set_ylabel("Required shaft power [W/kg]")
sec = ax.secondary_xaxis("top", functions=(wing_loading_g_dm2, wing_loading_n_m2))
sec.set_xlabel("Wing loading [g/dm^2]")
ax.set_title(f"Flying-wing matching chart, {alt_m} m ASL, eta_prop = {eta_p}")
ax.legend(loc="lower right", fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "01_matching_chart.png", dpi=140)
plt.show()


## Outputs handed to notebook 02


In [ ]:
constraints_summary = {
    "ws_limit_n_m2": float(ws_limit),
    "ws_limit_g_dm2": float(wing_loading_g_dm2(ws_limit)),
    "ws_design_n_m2": float(ws_design),
    "required_shaft_w_per_kg": p_design,
    "required_shaft_w": p_design * mass_kg,
    "min_wing_area_dm2_at_target_mass": mass_kg * G / ws_limit * 100,
}
constraints_summary
